# Spatial Anonymization – Usage Examples


This notebook demonstrates how to use the **spatial anonymization** library to protect survey GPS data while preserving analytical utility.


## Spatial Anonymization: Overview

### What Is Spatial Anonymization?

Spatial anonymization introduces controlled uncertainty into GPS coordinates to protect respondent privacy while preserving useful geographic patterns.

This library implements three common approaches:

- **DHS-style displacement**  
  Random displacement within a fixed maximum distance (optionally urban/rural-specific)

- **Donut geomasking**  
  Enforces a minimum displacement distance to reduce re-identification risk

- **Population-adaptive displacement**  
  Adjusts the displacement radius to ensure a minimum surrounding population

---

## Anonymization Logic

The anonymization process separates privacy policy, geometry, and validation:

1. **Radius Selection (Privacy Envelope)**  
   Defines the uncertainty range (fixed, donut, or population-based)

2. **Point Displacement (Geometry)**  
   Generates a candidate point using a random direction and distance

3. **Constraint Evaluation (Validity)**  
   Ensures the point satisfies spatial rules (e.g., boundaries, water exclusion)

Population defines the anonymization envelope, while constraints determine validity—ensuring predictable runtime, avoiding infinite retries, and enabling modular testing.

---

##  Dataset Description

This notebook uses household survey data from **Malawi** for demonstration purposes, including GPS coordinates and associated metadata.

- **Source:** Malawi Household Survey  
- **Coverage:** Multiple districts  
- **Unit:** Households within enumeration areas (EAs)

### Key Data Fields

- **Identifiers & geography:** `HHID`, `ea_id`, `ea_code`, `district`  
- **Survey metadata:** `interviewDate`, `reside`, `camp`, `hh_a02a`, `hh_a02b`  
- **GPS (Garmin – primary):** `latitude_garmin`, `longitude_garmin`  
- **GPS (Tablet – supplementary):** coordinates, accuracy, altitude, timestamp, weather, satellites  

### Data Characteristics
- Quality metrics support reliability checks  
- Urban/rural classification enables differentiated policies  

## 1. Python Setup <a id='setup'></a>

1. Ensure that you have the package installed by following instructions provided on the GitHub project documentation.
2. Import required libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

from spatial_anonymization.api import (
    anonymize_survey_coordinates, 
    validate_input_schema
    )
from spatial_anonymization.diagnostics import diagnostics, visualization

from spatial_anonymization.diagnostics.visualization import plot_displacement_comparison

# Import configuration presets from config_presets module
from spatial_anonymization.config.presets import DHS, print_preset

## 2. Working Directory and Inputs Setup

Define global path variables for your working directory and data folders. These paths are used to:
- load input datasets (e.g. survey coordinates, boundaries)  
- store downloaded resources (e.g. population rasters)  
- save outputs and diagnostics  

Centralizing paths improves clarity, portability, and reproducibility.  
Use a consistent naming convention: directories (`DIR_*`) and files (`FILE_*`).

### Example

```python
from pathlib import Path

# Base directory
DIR_WORK = Path("/path/to/your/project")

# Data directories
DIR_DATA = DIR_WORK / "data"
FILE_MWI_COORDS = DIR_WORK / "data/mwi/ihs_coords.dta"


In [3]:
# ======================================
# SETUP INPUT PATHS CONSTANTS
# ======================================
# Base working directory
DIR_WORK = Path("/Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/")

# Input data directories
DIR_DATA = DIR_WORK / "data"

# ========================
# INPUT FILES FOR MALAWI
# ========================
# MW working directory
DIR_MWI = DIR_DATA / "mwi"
DIR_MWI_GEOSPATIAL = DIR_MWI / "geospatial"

# Survey coordinates file
FILE_MWI_IHS6 = DIR_MWI / "test-coords.csv"


# MWI adm3 shapefile
FILE_MWI_ADM3 = DIR_MWI_GEOSPATIAL / "adm4-wgs84.shp"

# Pre-downloaded population raster file
FILE_MWI_POPULATION_RASTER = DIR_MWI_GEOSPATIAL / "mwi_worldpop_2025.tif"

# Pre-downloaded water bodies shapefile (from OSM)
FILE_MWI_WATER_BODIES = DIR_MWI_GEOSPATIAL / "MWI_water_bodies_osm.shp"

## 3. Input Data Preparation

Before running anonymization, users must prepare both coordinate data and any required geospatial inputs.

### 3.1 Coordinates

Ensure that the input dataset is cleaned and correctly structured:

- Longitude and latitude in decimal degrees (WGS84)  
- Numeric values with no missing entries  

Specify:
- longitude and latitude column names  
- urban/rural classification (if available)  
- labels for urban and rural areas  

---

### 3.2 Administrative Boundaries

Administrative boundaries are used to constrain displacement.

Users can:
- provide their own boundary data (e.g. shapefile or GeoJSON), or  
- allow the package to download boundaries automatically  

If downloading, users must specify:
- **country ISO code** (e.g. `MWI`)  
- **administrative level** (e.g. adm0–adm3)  

By default, the package attempts to download from:
- **[geoBoundaries](https://www.geoboundaries.org)**
- fallback to **[GADM](https://gadm.org/data.html)**

---

### 3.3 Population Data (Optional)

Population data is required only for population-based anonymization methods.

Users can:
- provide their own population raster, or  
- allow the package to download one automatically  

By default, the package uses:

- **[WorldPop (100m resolution, R2025A v1)](https://hub.worldpop.org/geodata/listing?id=135)**
  
---

In [10]:
# ============================================================
# SPECIFY INPUT DATA SCHEMA
# ============================================================
LAT = "latitude_tablet"
LON = "longitude_tablet"
URBAN_RURAL_COL = "urb_rur"
URBAN_VALUE = "URBAN"
RURAL_VALUE = "RURAL"

In [20]:
# ======================================
# VALIDATE INPUT DATA
# =======================================
# Load the data
df_mwi_ihs = pd.read_csv(FILE_MWI_IHS6)
print("=" * 50)
print("       Coordinates Data Preview")
print("=" * 50)
print(df_mwi_ihs[[LAT, LON, URBAN_RURAL_COL]].head())
print("-" * 50 )
# Check the data
print()
validate_input_schema(df_mwi_ihs, lat_col=LAT, lon_col=LON,
                      urban_rural_col=URBAN_RURAL_COL, urban_value=URBAN_VALUE,
                      rural_value=RURAL_VALUE)


       Coordinates Data Preview
   latitude_tablet  longitude_tablet urb_rur
0       -13.794762         32.893220   RURAL
1       -13.139925         33.797322   RURAL
2        -9.702232         33.274363   RURAL
3       -13.350432         33.904573   RURAL
4       -15.838955         34.394940   RURAL
--------------------------------------------------

All Input Data Schema Validations Passed!


In [7]:
# ======================================
# Geospatial Input Layers 
# =======================================
# Admin boundaries file defined in Working Directory and Inputs Setup
print(f"Admin boundaries file: {FILE_MWI_ADM3}")

# Population raster file defined in Working Directory and Inputs Setup
print(f"Population raster file: {FILE_MWI_POPULATION_RASTER}")

Admin boundaries file: /Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/mwi/geospatial/adm4-wgs84.shp
Population raster file: /Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/mwi/geospatial/mwi_worldpop_2025.tif


## 4. Configuring Spatial Anonymization

Before running anonymization, users need to make a few key decisions about how privacy should be enforced. These choices determine the displacement method, distance rules, and any spatial constraints.

The package supports two approaches:

### Option-1: Preset Configurations (Recommended)

Use predefined configurations for common methods such as DHS-style displacement.  
These presets reflect standard practices and require minimal setup, with optional overrides.

### Option-2. Custom Configuration

Define parameters manually for full control over:
- displacement method and distance thresholds  
- urban/rural rules  
- population thresholds  
- spatial constraints  

### Key Decisions

At a minimum, decide:
- which anonymization method to use  
- whether to differentiate urban and rural areas  
- whether to apply population thresholds  
- which constraints to enforce (e.g., administrative boundaries)

> **In this notebook, we begin with a vanilla DHS configuration**, which includes:
> - urban displacement up to 2 km  
> - rural displacement up to 5 km, with ~10% of points up to 10 km  
> - enforcement of administrative boundaries as the only spatial constraint  
>
> For administrative constraints, users must choose the appropriate boundary level.  
> In Malawi, administrative levels range from **adm0 (country)** to **adm3 (traditional authority)**, with enumeration areas being more granular.  
> In this example, displacement is constrained within **adm3 boundaries**.

In [21]:
# ======================================
# SPATIAL ANONYMIZATION CONFIGS 
# =======================================
# We will use the DHS preset configuration for this example, 
# which is suitable for survey data with urban/rural classification. The DHS preset includes parameters such as maximum displacement distances for urban and rural areas, and the number of random points to generate for each cluster.
METHOD = "dhs"

# Review the parameters for the DHS preset
print("DHS Preset Parameters:")
print_preset(DHS)

DHS Preset Parameters:

                        DHS Preset Parameters                         

  method: dhs
      Anonymization method to use. 'dhs' applies DHS-style random
      displacement, 'donut' enforces a minimum displacement distance,
      and 'population_adaptive' selects displacement distance based on
      a minimum population threshold.

  max_distance: 5,000
      Fallback maximum displacement distance (in meters) used when no
      urban/rural classification is available.

  urban_max_distance: 2,000
      Maximum displacement distance (in meters) applied to urban
      locations when urban/rural classification is provided.

  rural_max_distance: 5,000
      Maximum displacement distance (in meters) applied to rural
      locations when urban/rural classification is provided.

  rural_extra_max_distance: 10,000
      Larger maximum displacement distance (in meters) applied to a
      subset of rural locations to reduce re-identification risk
      (DHS-style rural ove

## 5. Running the Survey Anonymization

The `anonymize_survey_coordinates()` function is the main entry point for displacing GPS coordinates. It accepts the input DataFrame, coordinate column names, and the configuration parameters defined above.

### What happens internally

1. **Configuration resolution** — If no `config` object is passed, the function loads the preset matching the chosen `method` (e.g. DHS) and applies any parameter overrides.
2. **Input preparation** — Administrative boundaries (and optionally population rasters and water bodies) are loaded or auto-downloaded using the `country_iso3` code.
3. **Boundary filtering** — Points that fall outside the provided admin boundaries are flagged and removed before displacement.
4. **Displacement** — Each point is randomly displaced within its assigned radius. Candidate locations are validated against spatial constraints (e.g. must remain inside the same admin unit). If a candidate fails, the algorithm retries up to `max_attempts` times.
5. **Output** — The function returns the input DataFrame augmented with anonymized coordinate columns (`lon_anon`, `lat_anon`) and metadata (`distance_km`, `attempts`, `max_radius_assigned`). A CSV and GeoJSON of the results are saved to `output_dir`.

### Key parameters

| Parameter | Purpose |
|---|---|
| `method` | Anonymization method (`"dhs"`, `"donut"`, `"population_adaptive"`) |
| `admin_boundaries` | Shapefile or GeoDataFrame for boundary constraints |
| `output_dir` | Directory where output CSV and GeoJSON are saved |
| `n_workers` | Number of parallel processes (set to `1` for reproducibility) |
| `max_attempts` | Retry limit per point before marking as failed |
| `extract_displaced_population` | Whether to query population raster at displaced locations |

In [22]:
# ======================================
# ANONYMIZE THE DATA
# =======================================
df_anonymized = anonymize_survey_coordinates(
    df=df_mwi_ihs,
    lon_col=LON,
    lat_col=LAT,
    urban_rural_col=URBAN_RURAL_COL,
    urban_value=URBAN_VALUE,
    rural_value=RURAL_VALUE,
    method=METHOD,
    country_iso3="MWI",
    output_dir=DIR_MWI / "ihs6-2024-25",
    admin_boundaries=FILE_MWI_ADM3,
    water_boundaries=FILE_MWI_WATER_BODIES,
    population_raster=FILE_MWI_POPULATION_RASTER,
    n_workers=12,
    admin_level=3,
    max_attempts=400,
    extract_displaced_population=True,
    relax_admin_constraint_on_failure=False,
)

╔═════════════════════════════════════════════════════════════════════════╗
║                       ANONYMIZATION CONFIGURATION                       ║
╠═════════════════════════════════════════════════════════════════════════╣
║  Preset: dhs  (+4 overrides)                                            ║
╠═════════════════════════════════════════════════════════════════════════╣
║  DISPLACEMENT METHOD                                                    ║
║   method                   dhs          Random displacement within max radius║
╠═════════════════════════════════════════════════════════════════════════╣
║  DISTANCE THRESHOLDS                                                    ║
║   urban_max_distance       2000 m       Max displacement for urban points║
║   rural_max_distance       5000 m       Max displacement for rural points║
║   rural_extra_max          10000 m      Extended rural displacement     ║
║   rural_extra_prop         10.0%        Fraction of rural points extended║
║   

## 6. Review Anonymized Results

After anonymization, review outputs to verify quality and completeness.

### 6.1 Handling failed displacements

Points near tight boundaries may fail to find a valid location. Recommended workflow:

1. Start with `relax_admin_constraint_on_failure=False` (default) and assess failures  
2. Increase `max_attempts` (e.g. 100–300)  
3. If failures persist, set `relax_admin_constraint_on_failure=True`  
   - such points are flagged as `relaxed_admin=True`  

---

### 6.2 Output folder structure

Outputs are saved under `output_dir/anonymization_outputs/`:

| Folder | Contents |
|---|---|
| `processing_logs/` | `params.yaml` (config snapshot), `processing_log.txt` (run summary) |
| `intermediate_data/` | dropped and failed points |
| `output_data/` | `anonymized.csv`, `anonymized.geojson` |

---

### 6.3 Output DataFrame

The returned DataFrame includes:

| Column | Description |
|---|---|
| `lon_anon`, `lat_anon` | Displaced coordinates |
| `distance_km` | Displacement distance |
| `attempts` | Number of retries |
| `max_radius_assigned` | Assigned max radius (DHS rule) |
| `relaxed_admin` | Displaced outside admin constraint (if enabled) |
| `constraining_admin` | Admin level used |
| `pop_within_<r>m` | Population within radius (if enabled) |

---

### 6.4 Validation Checks

Review outputs using the following checks:

- **Dropped / failed points**  
  - dropped (outside boundaries) and failed points are saved in `intermediate_data/`  
  - compare input vs output row counts  

- **Displacement statistics**  
  - verify distances fall within expected ranges (e.g. DHS limits)  
  - summarize by urban/rural where applicable  

- **Spatial validation**  
  - plot original vs displaced points  
  - confirm points remain within intended boundaries  

- **Output files**  
  - verify `anonymized.csv` and `anonymized.geojson` exist and match expected counts  

In [ ]:
# ===================================================================
# DIAGNOSTICS: CHECK ANONYMIZED POINTS FALL WITHIN ADMIN BOUNDARIES
# ===================================================================
gdf_adm4 = gpd.read_file(FILE_MWI_ADM3)
admin_check = diagnostics.admin_boundary_check(df_anonymized, lon_col=LON, lat_col=LAT, admin_gdf=gdf_adm4)

[Admin Check] 4987/4987 displaced points stayed in same admin unit, 0 escaped (0.0%)


In [24]:
# ============================================
# DIAGNOSTICS: DISPLACEMENT STATISTICS
# ============================================
displacement_stats = diagnostics.displacement_statistics(
    df=df_anonymized,
    lon_col=LON,
    lat_col=LAT,
    urban_col=URBAN_RURAL_COL,
)


 Distance Stats: Original vs. Displaced Coordinates

--- all ---
  Stat    Distance (Km)
  ------  -------------
  count       4987.0000
  mean           2.8799
  median         2.8089
  min            0.0123
  max            9.9675
  p90            4.7308
  p95            4.9553
  p99            8.6670

--- RURAL ---
  Stat                 Distance (Km)
  -------------------  -------------
  count                    4140.0000
  mean                        3.2468
  median                      3.2522
  min                         0.0770
  max                         9.9675
  p90                         4.8157
  p95                         5.1680
  p99                         8.7834
  check_max<=10km             1.0000
  share_>5km                  0.0524
  share_>10km                 0.0000
  expected_share_>5km         0.0100

--- URBAN ---
  Stat            Distance (Km)
  --------------  -------------
  count                847.0000
  mean                   1.0866
  median          

In [ ]:
# ============================================
# VISUALIZE THE DISPLACEMENTS ON A MAP
# ============================================

# Load the admin boundaries for visualization
gdf_adm3 = gpd.read_file(FILE_MWI_ADM3)

# 
plot_displacement_comparison(
    df=df_anonymized,
    admin_gdf=gdf_adm3,
    n_buffers=10,
    lon_col=LON,
    lat_col=LAT,
    lon_anon_col="lon_anon",
    lat_anon_col="lat_anon",
    output_path=DIR_MWI / "ihs6-2024-25" / "displacement_map.html",
)

[Visualization] Sampled 500 of 4987 records (pass n_sample=None to plot all)
[Visualization] Map saved to /Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/mwi/ihs6-2024-25/displacement_map.html
